# Chapter 2, Session 2: MinIO — Your Local S3

**Goal:** Understand object storage by using it — same API you'd use with real AWS S3.

We'll:
1. Connect to MinIO using `boto3` (the standard AWS SDK)
2. Create buckets, upload/download objects
3. Understand the flat namespace — there are no real folders
4. Upload our FineWeb Parquet data and query it directly from object storage
5. Explore metadata and object versioning concepts

**MinIO is running at:**
- API: `http://localhost:9000` (where boto3 connects)
- Web Console: `http://localhost:9001` (open in browser: user=`minioadmin`, pass=`minioadmin`)

## Step 1: Connect to MinIO with boto3

`boto3` is the AWS SDK for Python. It talks to any S3-compatible storage.

The only difference from real S3: we pass `endpoint_url` to point at MinIO instead of AWS.

In [4]:
import boto3
from botocore.client import Config

# Connect to MinIO — identical to S3, just different endpoint
s3 = boto3.client(
    's3',
    endpoint_url='http://localhost:9000',       # MinIO instead of AWS
    aws_access_key_id='minioadmin',             # Default MinIO credentials
    aws_secret_access_key='minioadmin',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'                     # Required but doesn't matter locally
)

# Test the connection
response = s3.list_buckets()
print(f"Connected to MinIO!")
print(f"Existing buckets: {[b['Name'] for b in response['Buckets']]}")

Connected to MinIO!
Existing buckets: []


**What just happened:**
- `boto3.client('s3', ...)` creates an S3 client — same code you'd use for AWS
- `endpoint_url` is the only thing that makes this point to MinIO instead of AWS
- If you removed `endpoint_url` and used real AWS credentials, this exact code would talk to real S3

This is why MinIO is so useful for learning — **zero code changes when you move to production AWS**.

## Step 2: Create Buckets

A **bucket** is the top-level container. Think of it as a drive or volume.

Common pattern in LLM data engineering:
- `raw-data` — original crawled/downloaded data
- `processed-data` — cleaned, filtered, deduplicated
- `training-data` — final data ready for model training

In [5]:
# Create buckets for our LLM data pipeline
buckets = ['raw-data', 'processed-data', 'training-data']

for bucket in buckets:
    try:
        s3.create_bucket(Bucket=bucket)
        print(f"Created bucket: {bucket}")
    except s3.exceptions.BucketAlreadyOwnedByYou:
        print(f"Bucket already exists: {bucket}")

# Verify
response = s3.list_buckets()
print(f"\nAll buckets: {[b['Name'] for b in response['Buckets']]}")

Created bucket: raw-data
Created bucket: processed-data
Created bucket: training-data

All buckets: ['processed-data', 'raw-data', 'training-data']


## Step 3: Upload Objects — The Flat Namespace

Let's upload some text files to understand how keys work.

Remember: **there are no folders.** The key `2024/english/doc1.txt` is just a string with slashes in it.

In [6]:
# Upload some sample documents with "folder-like" keys
sample_docs = {
    'crawl-2024-01/english/article_001.txt': 'The quick brown fox jumps over the lazy dog.',
    'crawl-2024-01/english/article_002.txt': 'Machine learning is transforming data engineering.',
    'crawl-2024-01/german/artikel_001.txt': 'Der schnelle braune Fuchs springt über den faulen Hund.',
    'crawl-2024-02/english/article_001.txt': 'New data from February crawl.',
    'metadata/pipeline_config.json': '{"version": "1.0", "filters": ["length", "language"]}',
}

for key, content in sample_docs.items():
    s3.put_object(Bucket='raw-data', Key=key, Body=content.encode())
    print(f"Uploaded: raw-data/{key}")

print(f"\nUploaded {len(sample_docs)} objects")

Uploaded: raw-data/crawl-2024-01/english/article_001.txt
Uploaded: raw-data/crawl-2024-01/english/article_002.txt
Uploaded: raw-data/crawl-2024-01/german/artikel_001.txt
Uploaded: raw-data/crawl-2024-02/english/article_001.txt
Uploaded: raw-data/metadata/pipeline_config.json

Uploaded 5 objects


In [7]:
# List ALL objects in the bucket — see the flat namespace
response = s3.list_objects_v2(Bucket='raw-data')

print("=== All Objects (flat list — no real folders!) ===")
for obj in response.get('Contents', []):
    print(f"  Key: {obj['Key']:<55} Size: {obj['Size']:>5} bytes   Modified: {obj['LastModified']}")

=== All Objects (flat list — no real folders!) ===
  Key: crawl-2024-01/english/article_001.txt                   Size:    44 bytes   Modified: 2026-02-16 23:53:49.562000+00:00
  Key: crawl-2024-01/english/article_002.txt                   Size:    50 bytes   Modified: 2026-02-16 23:53:49.579000+00:00
  Key: crawl-2024-01/german/artikel_001.txt                    Size:    56 bytes   Modified: 2026-02-16 23:53:49.587000+00:00
  Key: crawl-2024-02/english/article_001.txt                   Size:    29 bytes   Modified: 2026-02-16 23:53:49.596000+00:00
  Key: metadata/pipeline_config.json                           Size:    53 bytes   Modified: 2026-02-16 23:53:49.604000+00:00


In [8]:
# "Browse" a folder — using Prefix to simulate directory listing
# This is how S3 console shows "folders" — it's just filtering by prefix
print("=== Simulating folder browse: crawl-2024-01/english/ ===")
response = s3.list_objects_v2(Bucket='raw-data', Prefix='crawl-2024-01/english/')
for obj in response.get('Contents', []):
    print(f"  {obj['Key']}")

print("\n=== Simulating top-level folders using Delimiter ===")
response = s3.list_objects_v2(Bucket='raw-data', Delimiter='/')
for prefix in response.get('CommonPrefixes', []):
    print(f"  📁 {prefix['Prefix']}")

=== Simulating folder browse: crawl-2024-01/english/ ===
  crawl-2024-01/english/article_001.txt
  crawl-2024-01/english/article_002.txt

=== Simulating top-level folders using Delimiter ===
  📁 crawl-2024-01/
  📁 crawl-2024-02/
  📁 metadata/


**Key insight above:**
- `Prefix='crawl-2024-01/english/'` — filters keys starting with that string. Not browsing a folder, just string matching.
- `Delimiter='/'` — groups keys by `/` to simulate top-level "folders". The API returns `CommonPrefixes` instead of actual directories.
- This is why S3 scales infinitely — no directory tree to maintain, just key string lookups.

## Step 4: Download and Read Objects

In [9]:
# Download a specific object
response = s3.get_object(Bucket='raw-data', Key='crawl-2024-01/english/article_001.txt')
content = response['Body'].read().decode()

print(f"Content: {content}")
print(f"Content-Type: {response['ContentType']}")
print(f"Content-Length: {response['ContentLength']} bytes")
print(f"Last-Modified: {response['LastModified']}")
print(f"ETag: {response['ETag']}")

Content: The quick brown fox jumps over the lazy dog.
Content-Type: binary/octet-stream
Content-Length: 44 bytes
Last-Modified: 2026-02-16 23:53:49+00:00
ETag: "e4d909c290d0fb1ca068ffaddf22cbd0"


**What's an ETag?** It's a hash of the object content. If two objects have the same ETag, they're identical. This is used for:
- Cache validation ("has this file changed since I last downloaded it?")
- Deduplication checks
- Multipart upload verification

## Step 5: Upload Real Data — FineWeb Parquet to Object Storage

Now let's do what real LLM data pipelines do: store Parquet files in object storage.

In [10]:
import os

# Upload our FineWeb Parquet files from Session 1
format_dir = '../data/format_comparison'
parquet_files = [f for f in os.listdir(format_dir) if f.endswith('.parquet')]

print("Uploading Parquet files to MinIO...")
for fname in parquet_files:
    local_path = os.path.join(format_dir, fname)
    key = f'fineweb/parquet/{fname}'
    size = os.path.getsize(local_path) / 1024 / 1024
    
    s3.upload_file(local_path, 'raw-data', key)
    print(f"  Uploaded: {key} ({size:.1f} MB)")

# Also upload the JSONL and CSV for comparison
for ext in ['jsonl', 'csv']:
    fname = f'fineweb.{ext}'
    local_path = os.path.join(format_dir, fname)
    if os.path.exists(local_path):
        key = f'fineweb/{ext}/{fname}'
        size = os.path.getsize(local_path) / 1024 / 1024
        s3.upload_file(local_path, 'raw-data', key)
        print(f"  Uploaded: {key} ({size:.1f} MB)")

print("\nDone!")

Uploading Parquet files to MinIO...
  Uploaded: fineweb/parquet/fineweb_lz4.parquet (9.9 MB)
  Uploaded: fineweb/parquet/fineweb_none.parquet (15.5 MB)
  Uploaded: fineweb/parquet/fineweb.parquet (9.5 MB)
  Uploaded: fineweb/parquet/fineweb_snappy.parquet (9.5 MB)
  Uploaded: fineweb/parquet/fineweb_zstd.parquet (6.8 MB)
  Uploaded: fineweb/parquet/fineweb_gzip.parquet (6.2 MB)
  Uploaded: fineweb/jsonl/fineweb.jsonl (16.9 MB)
  Uploaded: fineweb/csv/fineweb.csv (16.1 MB)

Done!


In [11]:
# List everything in the raw-data bucket now
response = s3.list_objects_v2(Bucket='raw-data')

print("=== raw-data bucket contents ===")
total_size = 0
for obj in response.get('Contents', []):
    size_mb = obj['Size'] / 1024 / 1024
    total_size += obj['Size']
    print(f"  {obj['Key']:<60} {size_mb:>8.2f} MB")

print(f"\nTotal: {total_size / 1024 / 1024:.1f} MB across {len(response['Contents'])} objects")

=== raw-data bucket contents ===
  crawl-2024-01/english/article_001.txt                            0.00 MB
  crawl-2024-01/english/article_002.txt                            0.00 MB
  crawl-2024-01/german/artikel_001.txt                             0.00 MB
  crawl-2024-02/english/article_001.txt                            0.00 MB
  fineweb/csv/fineweb.csv                                         16.07 MB
  fineweb/jsonl/fineweb.jsonl                                     16.86 MB
  fineweb/parquet/fineweb.parquet                                  9.54 MB
  fineweb/parquet/fineweb_gzip.parquet                             6.21 MB
  fineweb/parquet/fineweb_lz4.parquet                              9.87 MB
  fineweb/parquet/fineweb_none.parquet                            15.53 MB
  fineweb/parquet/fineweb_snappy.parquet                           9.54 MB
  fineweb/parquet/fineweb_zstd.parquet                             6.82 MB
  metadata/pipeline_config.json                                    

## Step 6: Read Parquet Directly from Object Storage

In real pipelines, you don't download files to local disk first — you read directly from S3.

PyArrow and Pandas can both do this.

In [12]:
import pyarrow.parquet as pq
import pyarrow.fs as pafs
import pandas as pd
import io
import time

# Method 1: Download to memory, then read with pandas
start = time.time()
response = s3.get_object(Bucket='raw-data', Key='fineweb/parquet/fineweb_snappy.parquet')
data = response['Body'].read()
df_from_s3 = pd.read_parquet(io.BytesIO(data))
method1_time = time.time() - start

print(f"Method 1 (boto3 + pandas): {len(df_from_s3)} rows in {method1_time:.3f}s")
print(f"Columns: {list(df_from_s3.columns)}")
print(f"\nFirst document preview:")
print(f"  URL: {df_from_s3.iloc[0]['url']}")
print(f"  Text: {df_from_s3.iloc[0]['text'][:150]}...")

Method 1 (boto3 + pandas): 5000 rows in 1.473s
Columns: ['text', 'id', 'dump', 'url', 'date', 'file_path', 'language', 'language_score', 'token_count']

First document preview:
  URL: http://daytimeroyaltyonline.com/single/?p=8906650&t=8780053
  Text: |Viewing Single Post From: Spoilers for the Week of February 11th|
|Lil||Feb 1 2013, 09:58 AM|
Don't care about Chloe/Taniel/Jen-Jen. Don't care about...


In [13]:
# Method 2: Read ONLY specific columns from S3 (column pruning over network!)
# This downloads less data from S3 — critical when your data is terabytes
start = time.time()
response = s3.get_object(Bucket='raw-data', Key='fineweb/parquet/fineweb_snappy.parquet')
data = response['Body'].read()
df_cols_only = pd.read_parquet(io.BytesIO(data), columns=['token_count', 'language_score', 'url'])
method2_time = time.time() - start

print(f"Method 2 (column pruning): {len(df_cols_only)} rows, 3 columns in {method2_time:.3f}s")
print(f"\nAvg token count: {df_cols_only['token_count'].mean():.0f}")
print(f"Avg language score: {df_cols_only['language_score'].mean():.3f}")

# Note: with real S3, PyArrow can do TRUE column pruning over the network
# (only downloading the column bytes, not the whole file)
# With boto3 get_object, we still download the full file.
# Spark/Ray do this efficiently using the S3A filesystem connector.

Method 2 (column pruning): 5000 rows, 3 columns in 0.058s

Avg token count: 689
Avg language score: 0.927


## Step 7: Simulate a Data Pipeline — Raw → Processed → Training

Let's simulate what a real pipeline does:
1. Read raw data from `raw-data` bucket
2. Apply a quality filter
3. Write the cleaned data to `processed-data` bucket
4. Select final training columns and write to `training-data` bucket

In [14]:
# Step 1: Read raw data from MinIO
response = s3.get_object(Bucket='raw-data', Key='fineweb/parquet/fineweb_snappy.parquet')
raw_df = pd.read_parquet(io.BytesIO(response['Body'].read()))
print(f"1. Read {len(raw_df)} raw documents from raw-data bucket")

# Step 2: Quality filter — keep docs with 50+ words and high language score
raw_df['word_count'] = raw_df['text'].str.split().str.len()
processed_df = raw_df[
    (raw_df['word_count'] >= 50) & 
    (raw_df['language_score'] >= 0.9)
].copy()
print(f"2. After quality filter: {len(processed_df)} documents ({len(processed_df)/len(raw_df)*100:.1f}% kept)")

# Step 3: Write processed data to processed-data bucket
buffer = io.BytesIO()
processed_df.to_parquet(buffer, compression='snappy')
buffer.seek(0)
s3.put_object(Bucket='processed-data', Key='fineweb/v1/processed.parquet', Body=buffer.getvalue())
print(f"3. Wrote processed data to processed-data/fineweb/v1/processed.parquet")

# Step 4: Select only training columns, write to training-data bucket
training_df = processed_df[['text', 'token_count']].copy()
buffer = io.BytesIO()
training_df.to_parquet(buffer, compression='zstd')  # ZSTD for final training data
buffer.seek(0)
s3.put_object(Bucket='training-data', Key='fineweb/v1/train.parquet', Body=buffer.getvalue())
print(f"4. Wrote training data to training-data/fineweb/v1/train.parquet")

# Summary
print(f"\n=== Pipeline Summary ===")
print(f"  Raw:       {len(raw_df)} docs")
print(f"  Processed: {len(processed_df)} docs (filtered)")
print(f"  Training:  {len(training_df)} docs, {training_df.columns.tolist()} columns only")

1. Read 5000 raw documents from raw-data bucket
2. After quality filter: 3930 documents (78.6% kept)
3. Wrote processed data to processed-data/fineweb/v1/processed.parquet
4. Wrote training data to training-data/fineweb/v1/train.parquet

=== Pipeline Summary ===
  Raw:       5000 docs
  Processed: 3930 docs (filtered)
  Training:  3930 docs, ['text', 'token_count'] columns only


In [15]:
# Verify — check all three buckets
for bucket_name in ['raw-data', 'processed-data', 'training-data']:
    response = s3.list_objects_v2(Bucket=bucket_name)
    contents = response.get('Contents', [])
    total = sum(obj['Size'] for obj in contents)
    print(f"\n📦 {bucket_name}: {len(contents)} objects, {total/1024/1024:.1f} MB")
    for obj in contents:
        print(f"   {obj['Key']:<55} {obj['Size']/1024/1024:>6.2f} MB")


📦 raw-data: 13 objects, 90.4 MB
   crawl-2024-01/english/article_001.txt                     0.00 MB
   crawl-2024-01/english/article_002.txt                     0.00 MB
   crawl-2024-01/german/artikel_001.txt                      0.00 MB
   crawl-2024-02/english/article_001.txt                     0.00 MB
   fineweb/csv/fineweb.csv                                  16.07 MB
   fineweb/jsonl/fineweb.jsonl                              16.86 MB
   fineweb/parquet/fineweb.parquet                           9.54 MB
   fineweb/parquet/fineweb_gzip.parquet                      6.21 MB
   fineweb/parquet/fineweb_lz4.parquet                       9.87 MB
   fineweb/parquet/fineweb_none.parquet                     15.53 MB
   fineweb/parquet/fineweb_snappy.parquet                    9.54 MB
   fineweb/parquet/fineweb_zstd.parquet                      6.82 MB
   metadata/pipeline_config.json                             0.00 MB

📦 processed-data: 1 objects, 8.2 MB
   fineweb/v1/processed.parquet  

**This is the real pattern:**

```
raw-data/          →  processed-data/       →  training-data/
(crawled web pages)   (cleaned + filtered)     (text + token_count only)
(all formats)         (Parquet, Snappy)         (Parquet, ZSTD compressed)
(biggest)             (smaller)                 (smallest)
```

Each bucket represents a stage. This separation lets you:
- Re-run processing without re-downloading raw data
- Version each stage independently
- Set different storage tiers (raw in Glacier, training in Standard)

## Step 8: Object Metadata and Tags

Every S3 object can carry custom metadata — useful for tracking data lineage without a separate database.

In [16]:
import json
from datetime import datetime

# Upload processed data WITH metadata
buffer = io.BytesIO()
processed_df.to_parquet(buffer, compression='snappy')
buffer.seek(0)

s3.put_object(
    Bucket='processed-data',
    Key='fineweb/v2/processed.parquet',
    Body=buffer.getvalue(),
    Metadata={
        'source': 'fineweb-sample-10BT',
        'pipeline-version': '1.0',
        'filters': 'word_count>=50,language_score>=0.9',
        'input-rows': str(len(raw_df)),
        'output-rows': str(len(processed_df)),
        'created-by': 'ch02-session2-notebook',
        'created-at': datetime.now().isoformat(),
    }
)
print("Uploaded with metadata!")

# Read the metadata back
response = s3.head_object(Bucket='processed-data', Key='fineweb/v2/processed.parquet')
print(f"\n=== Object Metadata ===")
print(f"Size: {response['ContentLength'] / 1024 / 1024:.1f} MB")
print(f"Last Modified: {response['LastModified']}")
print(f"\nCustom metadata:")
for key, value in response['Metadata'].items():
    print(f"  {key}: {value}")

Uploaded with metadata!

=== Object Metadata ===
Size: 8.2 MB
Last Modified: 2026-02-17 02:57:54+00:00

Custom metadata:
  created-at: 2026-02-16T20:57:54.603287
  created-by: ch02-session2-notebook
  filters: word_count>=50,language_score>=0.9
  input-rows: 5000
  output-rows: 3930
  pipeline-version: 1.0
  source: fineweb-sample-10BT


**Why this matters:** In Chapter 2's discussion of data lineage, the book says you need to track "where did the data come from." Object metadata is the simplest way — no extra database needed. You can always inspect any object and know exactly how it was produced.

## Summary

What you now know:

| Concept | What You Did |
|---------|-------------|
| **boto3 connection** | Same API for MinIO and real AWS S3 |
| **Buckets** | Created pipeline-stage buckets (raw, processed, training) |
| **Flat namespace** | Keys are just strings — `/` is cosmetic, not a directory |
| **Prefix filtering** | `Prefix=` simulates folder browsing |
| **Read from S3** | Load Parquet directly from object storage into pandas |
| **Data pipeline** | Raw → filter → processed → select columns → training |
| **Object metadata** | Attach lineage info to every object |

**Next session:** We'll put Apache Iceberg on top of this MinIO — adding ACID transactions, time travel, and schema evolution to our object storage.